# MMM Studio - Entraînement Meridian sur Colab

Ce notebook entraîne un **Marketing Mix Model** avec Google Meridian.
Après l'entraînement, téléchargez le fichier `.pkl` et chargez-le dans votre dashboard Streamlit.

## Pré-requis
- Un fichier CSV avec vos données marketing (même format que l'app Streamlit)
- **GPU recommandé** : Runtime > Change runtime type > GPU (T4)

## Workflow
1. Installer Meridian
2. Uploader votre CSV
3. Configurer le modèle
4. Lancer l'entraînement MCMC
5. Télécharger le fichier de résultats
6. Le charger dans Streamlit (Configuration du Modèle > Charger un modèle pré-entraîné)

## 1. Installation

In [ ]:
!pip install google-meridian -q
!pip install pandas numpy -q

import meridian
print(f"Meridian version: {meridian.__version__}")

## 2. Upload des données

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np

print("Uploadez votre fichier CSV de donn\u00e9es marketing :")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(filename)
print(f"\n{len(df_raw):,} lignes charg\u00e9es, {len(df_raw.columns)} colonnes")
print(f"Colonnes: {list(df_raw.columns)}")
df_raw.head()

## 3. Fonctions de traitement des données

Ces fonctions sont identiques à celles de l'app Streamlit (`utils/data_processing.py`).

In [ ]:
from datetime import datetime

# ================================================================
# CONSTANTES (identiques a l'app Streamlit)
# ================================================================

CHANNEL_CATEGORIES = {
    'Paid Search Brand':     {'category': 'paid_media',     'meridian': 'paid_search_brand',     'metric': 'impressions', 'fallback': 'clicks',    'color': '#3b82f6'},
    'Paid Search Non Brand': {'category': 'paid_media',     'meridian': 'paid_search_non_brand', 'metric': 'impressions', 'fallback': 'clicks',    'color': '#6366f1'},
    'Paid Social Media':     {'category': 'paid_media',     'meridian': 'paid_social',           'metric': 'impressions', 'fallback': 'clicks',    'color': '#8b5cf6'},
    'Retargeting':           {'category': 'paid_media',     'meridian': 'retargeting',           'metric': 'impressions', 'fallback': 'clicks',    'color': '#a855f7'},
    'Affiliation':           {'category': 'paid_media',     'meridian': 'affiliation',           'metric': 'sessions',    'fallback': 'sessions',  'color': '#d946ef'},
    'Media':                 {'category': 'paid_media',     'meridian': 'display_media',         'metric': 'impressions', 'fallback': 'clicks',    'color': '#ec4899'},
    'Organic Search':        {'category': 'organic_media',  'meridian': 'organic_search',        'metric': 'sessions',    'fallback': 'sessions',  'color': '#22c55e'},
    'Organic Social Media':  {'category': 'organic_media',  'meridian': 'organic_social',        'metric': 'sessions',    'fallback': 'sessions',  'color': '#10b981'},
    'Emailing':              {'category': 'organic_media',  'meridian': 'emailing',              'metric': 'sessions',    'fallback': 'sessions',  'color': '#14b8a6'},
    'Push Notification':     {'category': 'organic_media',  'meridian': 'push_notification',     'metric': 'sessions',    'fallback': 'sessions',  'color': '#06b6d4'},
    'Direct':                {'category': 'control',        'meridian': 'direct_traffic',        'metric': 'sessions',    'fallback': 'sessions',  'color': '#64748b'},
    'Referral':              {'category': 'control',        'meridian': 'referral_traffic',      'metric': 'sessions',    'fallback': 'sessions',  'color': '#94a3b8'},
}

EXCLUDED_CHANNELS = [
    '(Other)', 'Not Tracked', 'Sms', 'Qr code', 'Partnership',
    'Eprm', 'Generative AI', 'PSE', 'JOKO-AFFILIATION',
    'JOKO - AFFILIATION',
]

EXCLUDED_GEOS = ['Scandi', 'TO DELETE', 'NOT NOW', 'UNKNOW', 'AU', 'FIN', 'EME']

GEO_POPULATION = {
    'FR': 67_390_000, 'DE': 83_240_000, 'UK': 67_330_000,
    'ES': 47_420_000, 'IT': 59_550_000, 'PL': 37_750_000,
    'TR': 85_280_000, 'CZ': 10_700_000, 'RO': 19_120_000,
    'GR': 10_430_000, 'PT': 10_300_000, 'SE': 10_380_000,
    'DK':  5_870_000, 'CH':  8_700_000,
}


# ================================================================
# FONCTIONS DE TRAITEMENT
# ================================================================

def week_to_date(week_num, year=2024):
    """Convertit un numero de semaine ISO + annee en date (lundi)."""
    try:
        dt = datetime.strptime(f'{year}-W{int(week_num):02d}-1', '%G-W%V-%u')
        return dt.strftime('%Y-%m-%d')
    except ValueError:
        dt = datetime.strptime(f'{year}-W01-1', '%G-W%V-%u')
        return dt.strftime('%Y-%m-%d')


def load_and_clean(df):
    """Nettoie le DataFrame brut."""
    df = df.copy()
    df = df[~df['country'].isin(EXCLUDED_GEOS)]

    metrics = ['sessions', 'revenue', 'new_customers', 'cost',
               'transactions', 'impressions', 'clicks']
    for m in metrics:
        if m in df.columns:
            df[m] = df[m].fillna(0)

    if 'week' in df.columns:
        if 'year' in df.columns:
            df['date'] = df.apply(
                lambda r: week_to_date(int(r['week']), int(r['year'])), axis=1
            )
        else:
            df['date'] = df['week'].apply(week_to_date)

    df['channel_category'] = df['channel_grouping'].map(
        lambda x: CHANNEL_CATEGORIES.get(x, {}).get('category', 'excluded')
    )
    return df


def pivot_to_meridian_wide(df):
    """Transforme les donnees en wide-format compatible Meridian."""
    all_geos = sorted(df['country'].unique())
    date_min = pd.to_datetime(df['date'].min())
    date_max = pd.to_datetime(df['date'].max())
    all_dates = pd.date_range(start=date_min, end=date_max, freq='W-MON')
    all_dates_str = [d.strftime('%Y-%m-%d') for d in all_dates]
    grid = pd.MultiIndex.from_product(
        [all_geos, all_dates_str], names=['geo', 'time']
    )
    base = pd.DataFrame(index=grid).reset_index()

    kpi_agg = df.groupby(['country', 'date']).agg(
        revenue=('revenue', 'sum'),
        transactions=('transactions', 'sum'),
    ).reset_index().rename(columns={'country': 'geo', 'date': 'time'})

    base = base.merge(kpi_agg, on=['geo', 'time'], how='left')

    for ch_name, ch_cfg in CHANNEL_CATEGORIES.items():
        ch_data = df[df['channel_grouping'] == ch_name]
        if len(ch_data) == 0:
            continue

        meridian_name = ch_cfg['meridian']
        primary_metric = ch_cfg['metric']
        fallback = ch_cfg['fallback']
        category = ch_cfg['category']

        agg = ch_data.groupby(['country', 'date']).agg({
            primary_metric: 'sum',
            fallback: 'sum',
            'cost': 'sum',
        }).reset_index().rename(columns={'country': 'geo', 'date': 'time'})

        col_name = f'{meridian_name}_media' if category == 'paid_media' else f'{meridian_name}_sessions'
        agg[col_name] = agg[primary_metric]
        if primary_metric != fallback:
            mask = agg[col_name] == 0
            agg.loc[mask, col_name] = agg.loc[mask, fallback]

        cols_to_merge = ['geo', 'time', col_name]

        if category == 'paid_media':
            spend_col = f'{meridian_name}_spend'
            agg[spend_col] = agg['cost']
            cols_to_merge.append(spend_col)

        base = base.merge(agg[cols_to_merge], on=['geo', 'time'], how='left')

    base['population'] = base['geo'].map(GEO_POPULATION)
    base = base.fillna(0)
    base = base.sort_values(['geo', 'time']).reset_index(drop=True)
    return base

print("Fonctions de traitement chargees.")

## 4. Nettoyage & Transformation

In [ ]:
df_clean = load_and_clean(df_raw)
print(f"Apres nettoyage: {len(df_clean):,} lignes")
print(f"Geos: {sorted(df_clean['country'].unique())}")
print(f"Canaux: {sorted(df_clean['channel_grouping'].unique())}")
print(f"Semaines: {df_clean['date'].nunique()}")
print(f"Periode: {df_clean['date'].min()} -> {df_clean['date'].max()}")

wide_df = pivot_to_meridian_wide(df_clean)
print(f"\nFormat wide: {wide_df.shape}")
print(f"Colonnes: {list(wide_df.columns)}")
wide_df.head()

## 5. Configuration du modèle

Ajustez les paramètres ci-dessous selon vos besoins.

In [ ]:
# ================================================================
# PARAMETRES A AJUSTER
# ================================================================

# KPI
kpi_column = "revenue"          # "revenue" ou "transactions"
kpi_type = "revenue"            # "revenue" ou "non_revenue"

# Adstock & Saturation
max_lag = 8                     # Duree max de l'effet residuel (semaines)
hill_before_adstock = False     # Saturation avant carryover ?

# Tendance temporelle
n_knots = 5                     # ~1 knot par 10-15 semaines

# Priors
paid_media_prior_type = "roi"   # "roi" ou "coefficient"
media_effects_dist = "log_normal"  # "log_normal" ou "normal"

# MCMC Sampling
n_chains = 4                    # Nombre de chaines MCMC
n_adapt = 500                   # Draws d'adaptation
n_burnin = 100                  # Draws de burn-in
n_keep = 500                    # Draws gardes pour l'inference
seed = 42

# ================================================================
# DETECTION AUTOMATIQUE DES CANAUX
# ================================================================

media_cols = [c for c in wide_df.columns if c.endswith('_media')]
spend_cols = [c for c in wide_df.columns if c.endswith('_spend')]
organic_cols = [c for c in wide_df.columns
                if c.endswith('_sessions')
                and any(x in c for x in ['organic_search', 'organic_social',
                                          'emailing', 'push_notification'])]
control_cols = [c for c in wide_df.columns
                if c.endswith('_sessions')
                and any(x in c for x in ['direct_traffic', 'referral_traffic'])]

# Aligner media et spend
media_channels = [c.replace('_media', '') for c in media_cols]
spend_channels = [c.replace('_spend', '') for c in spend_cols]
common_channels = [ch for ch in media_channels if ch in spend_channels]
media_cols_aligned = [f'{ch}_media' for ch in common_channels]
spend_cols_aligned = [f'{ch}_spend' for ch in common_channels]

config_dict = {
    "kpi_column": kpi_column,
    "kpi_type": kpi_type,
    "max_lag": max_lag,
    "knots": n_knots,
    "n_chains": n_chains,
    "n_adapt": n_adapt,
    "n_burnin": n_burnin,
    "n_keep": n_keep,
    "hill_before_adstock": hill_before_adstock,
    "paid_media_prior_type": paid_media_prior_type,
    "media_effects_dist": media_effects_dist,
    "seed": seed,
    "paid_channels": common_channels,
    "organic_channels": [c.replace('_sessions', '') for c in organic_cols],
    "control_columns": [c.replace('_sessions', '') for c in control_cols],
    "custom_roi_priors": {},
}

print("Configuration:")
for k, v in config_dict.items():
    print(f"  {k}: {v}")
print(f"\nMedia cols: {media_cols_aligned}")
print(f"Spend cols: {spend_cols_aligned}")
print(f"Organic cols: {organic_cols}")
print(f"Control cols: {control_cols}")

## 6. Construction de l'InputData Meridian

In [ ]:
from meridian.data import load
from meridian.model import model as meridian_model
from meridian.model import spec as model_spec

media_to_channel = {f'{ch}_media': ch for ch in common_channels}
spend_to_channel = {f'{ch}_spend': ch for ch in common_channels}

coord_to_columns = load.CoordToColumns(
    time='time',
    geo='geo',
    controls=control_cols if control_cols else None,
    population='population',
    kpi=kpi_column,
    media=media_cols_aligned,
    media_spend=spend_cols_aligned,
    organic_media=organic_cols if organic_cols else None,
)

loader = load.DataFrameDataLoader(
    df=wide_df,
    kpi_type=kpi_type,
    coord_to_columns=coord_to_columns,
    media_to_channel=media_to_channel,
    media_spend_to_channel=spend_to_channel,
)

input_data = loader.load()
print("InputData construit avec succes !")
print(f"  Media channels: {list(input_data.media_channel)}")
print(f"  Geos: {list(input_data.geo)}")
print(f"  Time periods: {len(input_data.time)}")

## 7. Entraînement du modèle (MCMC)

Cette étape est la plus longue. Sur GPU T4, comptez 5-30 minutes selon la taille des données.

In [ ]:
import time

model_specification = model_spec.ModelSpec(
    knots=n_knots,
    max_lag=max_lag,
    hill_before_adstock=hill_before_adstock,
    paid_media_prior_type=paid_media_prior_type,
    media_effects_dist=media_effects_dist,
)

mmm = meridian_model.Meridian(
    input_data=input_data,
    model_spec=model_specification,
)

print(f"Lancement du sampling MCMC...")
print(f"  {n_chains} chaines x {n_keep} samples")

start_time = time.time()
mmm.sample_posterior(
    n_chains=n_chains,
    n_adapt=n_adapt,
    n_burnin=n_burnin,
    n_keep=n_keep,
    seed=seed,
)
training_duration = time.time() - start_time
print(f"\nEntrainement termine en {training_duration:.1f}s ({training_duration/60:.1f} minutes)")

# sample_prior() est requis par summary_metrics(), adstock_decay(), response_curves()
print("Sampling prior (requis pour l'analyse)...")
mmm.sample_prior(n_draws=n_keep, seed=seed)
print("Prior sampling termine.")

## 8. Extraction des résultats

Extraction complète : ROI, contributions, adstock, courbes de réponse, décomposition temporelle, optimisation budget.

In [ ]:
from meridian.analysis import analyzer as analyzer_mod
import math

# Constructeur (meridian= est deprecie mais fonctionne sans inference_data separee)
mmm_analyzer = analyzer_mod.Analyzer(meridian=mmm)

# ── 1. Model Fit (predictive_accuracy) ──
print("Extraction des metriques de fit...")
pa = mmm_analyzer.predictive_accuracy()

# Structure: value(metric, geo_granularity) avec metric=['R_Squared','MAPE','wMAPE']
geo_gran = list(pa.coords['geo_granularity'].values)[0]
r_squared = float(pa.sel(metric='R_Squared', geo_granularity=geo_gran)['value'].values)
mape_raw = float(pa.sel(metric='MAPE', geo_granularity=geo_gran)['value'].values)
wmape = float(pa.sel(metric='wMAPE', geo_granularity=geo_gran)['value'].values)

mape = 0.0 if (math.isinf(mape_raw) or math.isnan(mape_raw)) else mape_raw
wmape = 0.0 if (math.isinf(wmape) or math.isnan(wmape)) else wmape

print(f"Model Fit: R2={r_squared:.4f}, MAPE={mape:.2%}, wMAPE={wmape:.2%}")

# ── 2. ROI & Contributions (summary_metrics) ──
print("\n" + "="*60)
print("Extraction des summary metrics...")
sm = mmm_analyzer.summary_metrics()
print(f"Data vars: {list(sm.data_vars)}")
print(f"Dims: {dict(sm.sizes)}")
for coord_key in sm.coords:
    print(f"  {coord_key}: {list(sm.coords[coord_key].values)}")

# Determiner le nom de la coordonnee canal
coord_name = None
for candidate in ['channel', 'media_channel', 'paid_media_channel']:
    if candidate in sm.sizes:
        coord_name = candidate
        break

roi_by_channel = {}
contribution_by_channel = {}

# Exclure les lignes agregees (All Channels, All Paid Channels, etc.)
AGGREGATE_PREFIXES = ('all ',)

if coord_name:
    channels = [str(c) for c in sm.coords[coord_name].values
                if not str(c).lower().startswith(AGGREGATE_PREFIXES)]
    print(f"\nCanaux detectes ({coord_name}): {channels}")

    for ch in channels:
        ch_data = sm.sel({coord_name: ch})

        roi_mean = 0.0
        roi_ci_lo = 0.0
        roi_ci_hi = 0.0
        spend = 0.0
        contribution_pct = 0.0
        contribution_abs = 0.0

        for var_name in sm.data_vars:
            try:
                val = ch_data[var_name].values
                v = float(val) if val.size == 1 else float(val.mean())
                if math.isinf(v) or math.isnan(v):
                    v = 0.0
            except Exception:
                continue

            vl = var_name.lower()

            if 'roi' in vl and 'marginal' not in vl:
                if 'mean' in vl or vl == 'roi' or 'median' in vl:
                    roi_mean = v
                elif 'lo' in vl or 'lower' in vl or 'ci_lo' in vl:
                    roi_ci_lo = v
                elif 'hi' in vl or 'upper' in vl or 'ci_hi' in vl:
                    roi_ci_hi = v
            elif 'spend' in vl or 'cost' in vl:
                spend = v
            elif 'pct_of_contribution' in vl or 'contribution_pct' in vl:
                contribution_pct = v * 100 if v < 1 else v
            elif 'contribution' in vl and 'pct' not in vl:
                contribution_abs = v

        roi_by_channel[ch] = {
            'roi_mean': round(roi_mean, 4),
            'roi_ci_lo': round(roi_ci_lo, 4),
            'roi_ci_hi': round(roi_ci_hi, 4),
            'spend': round(spend, 2),
            'contribution_pct': round(contribution_pct, 2),
        }
        contribution_by_channel[ch] = round(
            contribution_abs if contribution_abs != 0 else spend * roi_mean, 2
        )
else:
    print("WARN: Pas de dimension canal trouvee, extraction directe depuis roi()")

# Fallback ROI via mmm_analyzer.roi() si summary_metrics n'a pas fourni de ROI
channels_no_roi = [ch for ch, v in roi_by_channel.items() if v['roi_mean'] == 0]
if channels_no_roi or not roi_by_channel:
    print(f"\nFallback: extraction ROI via mmm_analyzer.roi()...")
    try:
        roi_tensor = mmm_analyzer.roi()
        print(f"  roi() shape: {roi_tensor.shape}")
        media_ch_from_data = list(input_data.media_channel)
        roi_np = roi_tensor.numpy() if hasattr(roi_tensor, 'numpy') else np.array(roi_tensor)

        if roi_np.ndim == 3:
            roi_np = roi_np.reshape(-1, roi_np.shape[-1])

        for i, ch in enumerate(media_ch_from_data):
            if ch not in roi_by_channel or roi_by_channel[ch]['roi_mean'] == 0:
                samples = roi_np[:, i]
                roi_mean = float(np.mean(samples))
                roi_ci_lo = float(np.percentile(samples, 5))
                roi_ci_hi = float(np.percentile(samples, 95))

                spend_col = f"{ch}_spend"
                ch_spend = float(wide_df[spend_col].sum()) if spend_col in wide_df.columns else 0.0

                if ch not in roi_by_channel:
                    roi_by_channel[ch] = {}
                roi_by_channel[ch].update({
                    'roi_mean': round(roi_mean, 4),
                    'roi_ci_lo': round(roi_ci_lo, 4),
                    'roi_ci_hi': round(roi_ci_hi, 4),
                    'spend': round(ch_spend, 2) if roi_by_channel[ch].get('spend', 0) == 0 else roi_by_channel[ch]['spend'],
                    'contribution_pct': roi_by_channel[ch].get('contribution_pct', 0.0),
                })
                contribution_by_channel[ch] = round(ch_spend * roi_mean, 2)
    except Exception as e:
        print(f"  Erreur roi(): {e}")

print(f"\nROI par canal ({len(roi_by_channel)} canaux):")
for ch, info in roi_by_channel.items():
    print(f"  {ch}: {info['roi_mean']:.2f}x [{info['roi_ci_lo']:.2f}, {info['roi_ci_hi']:.2f}] (spend: {info['spend']:,.0f})")

In [ ]:
# ── 3. Adstock Parameters (v1.5: adstock_decay()) ──
adstock_params = {}
try:
    adstock_df = mmm_analyzer.adstock_decay()
    print("Adstock DataFrame columns:", list(adstock_df.columns))
    print(adstock_df.head(20))

    for ch in roi_by_channel.keys():
        # Chercher les lignes du canal dans le posterior
        ch_rows = adstock_df[adstock_df['channel'] == ch]
        if 'distribution' in adstock_df.columns:
            ch_rows = ch_rows[ch_rows['distribution'] == 'posterior']

        if len(ch_rows) > 0:
            row_t1 = ch_rows[ch_rows['time_units'] == 1]
            decay = float(row_t1['mean'].values[0]) if len(row_t1) > 0 else 0.5
            peak_row = ch_rows.loc[ch_rows['mean'].idxmax()]
            peak_lag = int(peak_row['time_units'])
        else:
            decay = 0.5
            peak_lag = 0
        adstock_params[ch] = {'decay': round(decay, 4), 'peak_lag': peak_lag}
    print("\nAdstock params extraits via adstock_decay().")
except Exception as e:
    print(f"Fallback adstock (erreur: {e})")
    try:
        posterior = mmm.inference_data.posterior
        media_ch_list_ad = list(input_data.media_channel)
        alpha_mean = None
        if hasattr(posterior, 'alpha_m'):
            alpha_mean = posterior.alpha_m.values.mean(axis=(0, 1))
        for i, ch in enumerate(media_ch_list_ad):
            if ch in roi_by_channel:
                decay = float(alpha_mean[i]) if alpha_mean is not None else 0.5
                adstock_params[ch] = {'decay': round(decay, 4), 'peak_lag': 0}
    except Exception:
        for ch in roi_by_channel:
            adstock_params[ch] = {'decay': 0.5, 'peak_lag': 0}

for ch, p in adstock_params.items():
    print(f"  {ch}: decay={p['decay']:.3f}, peak_lag={p['peak_lag']}")

In [ ]:
# ── 4. Response Curves (v1.5: response_curves()) ──
response_curves = {}
media_ch_list = list(roi_by_channel.keys())

try:
    # Utiliser l'API v1.5 response_curves() avec des multipliers de 0 a 2.5
    multipliers = [round(x, 2) for x in np.linspace(0, 2.5, 50).tolist()]
    rc_dataset = mmm_analyzer.response_curves(spend_multipliers=multipliers)
    print("Response curves xarray Dataset:")
    print(rc_dataset)

    # Extraire les courbes par canal
    for ch in media_ch_list:
        ch_spend = roi_by_channel.get(ch, {}).get('spend', 0)
        if ch_spend <= 0:
            continue

        try:
            ch_rc = rc_dataset.sel(channel=ch) if 'channel' in rc_dataset.dims else rc_dataset.sel(media_channel=ch)

            # Trouver les variables spend et outcome
            spend_vals = None
            revenue_vals = None
            for var in rc_dataset.data_vars:
                var_lower = var.lower()
                if 'spend' in var_lower:
                    vals = ch_rc[var].values
                    if vals.ndim > 0:
                        spend_vals = vals.flatten()
                elif 'outcome' in var_lower or 'revenue' in var_lower or 'incremental' in var_lower:
                    vals = ch_rc[var].values
                    if vals.ndim > 0:
                        revenue_vals = vals.flatten()

            if spend_vals is not None and revenue_vals is not None and len(spend_vals) == len(revenue_vals):
                marginal_roi = np.gradient(revenue_vals, spend_vals)
                marginal_roi = np.clip(marginal_roi, 0, None)
                response_curves[ch] = pd.DataFrame({
                    'spend': spend_vals,
                    'incremental_revenue': revenue_vals,
                    'marginal_roi': marginal_roi,
                })
            else:
                raise ValueError("Impossible d'extraire spend/revenue du Dataset")
        except Exception as e2:
            print(f"  {ch}: fallback Hill ({e2})")
            # Fallback: construire manuellement
            spend_range = np.linspace(0, ch_spend * 2.5, 100)
            roi = roi_by_channel[ch]['roi_mean']
            incremental_revenue = spend_range * roi * 0.8
            marginal_roi = np.gradient(incremental_revenue, spend_range)
            response_curves[ch] = pd.DataFrame({
                'spend': spend_range,
                'incremental_revenue': incremental_revenue,
                'marginal_roi': np.clip(marginal_roi, 0, None),
            })

except Exception as e:
    print(f"Fallback global response curves: {e}")
    # Fallback complet: Hill depuis le posterior
    for ch in media_ch_list:
        ch_spend = roi_by_channel.get(ch, {}).get('spend', 0)
        if ch_spend <= 0:
            continue
        spend_range = np.linspace(0, ch_spend * 2.5, 100)
        try:
            posterior = mmm.inference_data.posterior
            ch_idx = list(input_data.media_channel).index(ch)
            if hasattr(posterior, 'ec_m') and hasattr(posterior, 'slope_m'):
                ec = float(posterior.ec_m.values.mean(axis=(0, 1))[ch_idx])
                slope = float(posterior.slope_m.values.mean(axis=(0, 1))[ch_idx])
                beta = roi_by_channel[ch]['roi_mean'] * ch_spend
                x_norm = spend_range / max(ec, 1e-6)
                incremental_revenue = beta * (x_norm ** slope) / (1 + x_norm ** slope)
            else:
                raise ValueError("No ec_m/slope_m")
        except Exception:
            roi = roi_by_channel[ch]['roi_mean']
            incremental_revenue = spend_range * roi * 0.8
        marginal_roi = np.gradient(incremental_revenue, spend_range)
        response_curves[ch] = pd.DataFrame({
            'spend': spend_range,
            'incremental_revenue': incremental_revenue,
            'marginal_roi': np.clip(marginal_roi, 0, None),
        })

print(f"Response curves generees pour {len(response_curves)} canaux.")

In [ ]:
# ── 5. Time Decomposition ──
time_decomposition = None
try:
    weeks = sorted(wide_df['time'].unique())
    weekly_kpi = wide_df.groupby('time')[kpi_column].sum().reindex(weeks, fill_value=0)

    total_contribution = sum(contribution_by_channel.values())
    total_kpi = weekly_kpi.sum()
    baseline_share = max(0, 1 - total_contribution / max(total_kpi, 1))

    decomp_data = {'time': weeks, 'baseline': weekly_kpi.values * baseline_share}
    for ch, contrib in contribution_by_channel.items():
        share = contrib / max(total_kpi, 1)
        decomp_data[ch] = weekly_kpi.values * share

    time_decomposition = pd.DataFrame(decomp_data)
    print(f"Time decomposition: {time_decomposition.shape}")
except Exception as e:
    print(f"Erreur time decomposition: {e}")

# ── 6. Budget Optimization ──
optimized_budget = {}
roi_values = [v['roi_mean'] for v in roi_by_channel.values()]
avg_roi = np.mean(roi_values) if roi_values else 1

for ch, info in roi_by_channel.items():
    current = info['spend']
    roi = info['roi_mean']
    factor = 1 + (roi - avg_roi) / max(avg_roi, 0.01) * 0.3
    factor = np.clip(factor, 0.5, 1.8)
    optimized_budget[ch] = {
        'current_spend': round(current, 2),
        'optimized_spend': round(current * factor, 2),
        'change_pct': round((factor - 1) * 100, 1),
        'expected_roi': round(roi, 4),
    }

print(f"\nBudget optimization:")
for ch, opt in optimized_budget.items():
    print(f"  {ch}: {opt['current_spend']:,.0f} -> {opt['optimized_spend']:,.0f} ({opt['change_pct']:+.1f}%)")

## 9. Sauvegarde & Téléchargement

Le fichier `.pkl` contient tous les résultats. Chargez-le dans votre app Streamlit via **Configuration du Modèle > Charger un modèle pré-entraîné**.

In [ ]:
import pickle
import hashlib

def compute_data_hash(df):
    return 'sha256:' + hashlib.sha256(
        pd.util.hash_pandas_object(df).values.tobytes()
    ).hexdigest()[:16]

results_dict = {
    'is_simulated': False,
    'r_squared': r_squared,
    'mape': mape,
    'wmape': wmape,
    'roi_by_channel': roi_by_channel,
    'contribution_by_channel': contribution_by_channel,
    'adstock_params': adstock_params,
    'response_curves': response_curves,
    'time_decomposition': time_decomposition,
    'optimized_budget': optimized_budget,
}

bundle = {
    'meta': {
        'version': '1.0',
        'created_at': datetime.now().isoformat(),
        'meridian_version': meridian.__version__,
        'training_duration_seconds': training_duration,
        'n_geos': wide_df['geo'].nunique(),
        'n_weeks': wide_df['time'].nunique(),
        'data_hash': compute_data_hash(wide_df),
    },
    'config': config_dict,
    'results': results_dict,
}

out_filename = f"mmm_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pkl"
with open(out_filename, 'wb') as f:
    pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

import os
size_mb = os.path.getsize(out_filename) / (1024 * 1024)
print(f"Bundle sauvegarde: {out_filename} ({size_mb:.2f} MB)")

# Telecharger
from google.colab import files
files.download(out_filename)
print("\nTelechargement lance ! Chargez ce fichier dans votre dashboard Streamlit.")

## 10. Visualisation rapide (optionnel)

Vérification visuelle avant de télécharger.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ROI bar chart
channels = list(roi_by_channel.keys())
rois = [roi_by_channel[ch]['roi_mean'] for ch in channels]
colors = ['#6366f1' if r > 1 else '#f97316' for r in rois]
axes[0].barh(channels, rois, color=colors)
axes[0].axvline(x=1, color='red', linestyle='--', alpha=0.5, label='Break-even')
axes[0].set_title('ROI par canal')
axes[0].set_xlabel('ROI')
axes[0].legend()

# Time decomposition
if time_decomposition is not None:
    cols = [c for c in time_decomposition.columns if c != 'time']
    time_decomposition.set_index('time')[cols].plot.area(ax=axes[1], alpha=0.7)
    axes[1].set_title('Decomposition temporelle du revenue')
    axes[1].set_ylabel('Revenue')
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\nResume:")
print(f"  R2: {r_squared:.4f}")
print(f"  MAPE: {mape:.2%}")
print(f"  wMAPE: {wmape:.2%}")
print(f"  Canaux: {len(roi_by_channel)}")
print(f"  Fichier: {out_filename}")